# Total cVeg annual analysis

Calculate annual total vegetation carbon stock Vegetation carbon (cVeg) over 60–90°N, map Exp1 and Exp4 to their GWL trajectories, and save compact plot data outside the repository. No figures or raw monthly fields are written to the GitHub project.

Add models in `MODEL_CONFIGS`. NASA-GISS model uses the cVeg combined file.

In [1]:
from pathlib import Path
import sys

# Locate the project from the notebook's working directory.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils" / "common.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project folder containing utils/common.py was not found.")
    PROJECT_ROOT = PROJECT_ROOT.parent

UTILS_DIR = PROJECT_ROOT / "utils"
LEGACY_CODE_DIR = PROJECT_ROOT / "code"

sys.path.insert(0, str(LEGACY_CODE_DIR))
sys.path.insert(0, str(UTILS_DIR))

from common import (
    find_project_root,
    load_ec_land_area,
    map_annual_to_gwl,
    remap_cell_area,
    remap_land_area_to_esm,
)
from cveg import annual_total_carbon
PROJECT_ROOT = find_project_root(PROJECT_ROOT)
import xarray as xr

MODEL_CONFIGS = [
    {"name": "EC-Earth3-ESM-1", "family": "EC", "member": "r1i1p1f1"},
    {
        "name": "GISS-E2-1-G-CC2",
        "family": "NASA",
        "member": "r1i1p1f3",
        "single_file": True,
        "area_method": "nasa_areacella",
    },
    {"name": "UKESM1-2-LL", "family": "UKESM", "member": "r1i1p1f1"},
]
EXPERIMENTS = ("Exp1", "Exp2", "Exp4")
GWL_LEGS = {"Exp1": "ramp-up", "Exp4": "ramp-down-2c"}
LAT_MIN, LAT_MAX = 60.0, 90.0

# Keep derived plot data outside the GitHub repository and raw-data folders.
OUTPUT_DIR = PROJECT_ROOT.parent / f"{PROJECT_ROOT.name}_plot_data" / "cveg"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def cveg_input_files(config, experiment):
    """Return the configured experiment's cVeg NetCDF files."""
    experiment_dir = PROJECT_ROOT / f"{experiment}_data"
    family = config["family"]
    if config.get("single_file", False):
        filename = config.get("file_overrides", {}).get(
            experiment, f"{experiment}_{family}_cVeg.nc"
        )
        return [experiment_dir / family / filename]

    folder_name = config.get("folder_overrides", {}).get(
        experiment, f"{experiment}_{family}_cVeg"
    )
    folder = experiment_dir / family / folder_name
    return sorted(folder.glob("*.nc"))

In [2]:
ec_land_area = load_ec_land_area(PROJECT_ROOT)
nasa_area_file = (
    PROJECT_ROOT / "fx"
    / "areacella_APfx_EC-Earth3-ESM-1_esm-up2p0_r1i1p1f1_gr.nc"
)
land_areas = {}

for config in MODEL_CONFIGS:
    model, family = config["name"], config["family"]
    sample_file = cveg_input_files(config, "Exp1")[0]
    with xr.open_dataset(sample_file) as sample:
        target_lat = sample["lat"].load().values
        target_lon = sample["lon"].load().values

    area_method = config.get("area_method", "ec_land_area" if family == "EC" else "remap_ec_land_area")
    if area_method == "ec_land_area":
        land_areas[model] = ec_land_area
    elif area_method == "nasa_areacella":
        with xr.open_dataset(nasa_area_file) as source:
            source_area = source["areacella"].load()
            lat_bounds = source["lat_bnds"].load().values
            lon_bounds = source["lon_bnds"].load().values
        land_areas[model] = remap_cell_area(
            source_area, lat_bounds, lon_bounds, target_lat, target_lon
        )
    else:
        land_areas[model] = remap_land_area_to_esm(
            ec_land_area, target_lat, target_lon
        )

    print(f"{model} area grid: {land_areas[model].sizes['lat']} × {land_areas[model].sizes['lon']}")

EC-Earth3-ESM-1 area grid: 256 × 512
GISS-E2-1-G-CC2 area grid: 90 × 144
UKESM1-2-LL area grid: 144 × 192


## Annual values and plot-data output 
Monthly cVeg density is weighted by days per month to form annual means, multiplied by the model's land-area grid, and summed over 60–90°N. Save annual Exp1/Exp2/Exp4 series, GWL-mapped Exp1/Exp4 series, and the Exp2 full-record mean. The output contains regional annual summaries only.

In [3]:
for config in MODEL_CONFIGS:
    model, family = config["name"], config["family"]
    annual = {}

    for experiment in EXPERIMENTS:
        files = cveg_input_files(config, experiment)
        annual[experiment] = annual_total_carbon(
            files,
            land_areas[model],
            variable="cVeg",
            lat_min=LAT_MIN,
            lat_max=LAT_MAX,
        )
        print(
            f"{model} {experiment}: {annual[experiment].sizes['year']} years; "
            f"mean={float(annual[experiment].mean('year')):.3f} Pg C"
        )

    mapped = {
        experiment: map_annual_to_gwl(
            annual[experiment], model, config["member"], leg, PROJECT_ROOT
        )
        for experiment, leg in GWL_LEGS.items()
    }

    plot_data = xr.Dataset()
    for experiment, series in annual.items():
        plot_data[f"{experiment.lower()}_annual"] = series.rename(
            {"year": f"year_{experiment.lower()}"}
        )
    for experiment, series in mapped.items():
        rename = {"gwl": f"gwl_{experiment.lower()}"}
        if "year" in series.coords:
            rename["year"] = f"source_year_{experiment.lower()}"
        plot_data[f"{experiment.lower()}_gwl"] = series.rename(rename)

    plot_data["exp2_full_record_mean"] = annual["Exp2"].mean("year")
    plot_data.attrs.update({
        "model": model,
        "dataset_family": family,
        "latitude_band": f"{LAT_MIN:g}–{LAT_MAX:g}°N",
        "units": "Pg C",
        "annual_method": (
            "Monthly grid-cell cVeg density was day-weighted to annual means, "
            "multiplied by grid-cell area, and summed over the latitude band."
        ),
        "description": (
            "Annual regional cVeg and GWL series for plotting; raw monthly "
            "spatial inputs are not included."
        ),
    })

    filename = model.replace("/", "_").replace(" ", "_") + "_cveg_plot_data.nc"
    output_file = OUTPUT_DIR / filename
    plot_data.to_netcdf(output_file)
    print(f"Saved plot data: {output_file}")

EC-Earth3-ESM-1 Exp1: 350 years; mean=53.739 Pg C
EC-Earth3-ESM-1 Exp2: 500 years; mean=43.080 Pg C
EC-Earth3-ESM-1 Exp4: 120 years; mean=28.276 Pg C
Saved plot data: /mnt/e/lund/NGEA52/hysteresis/tipmip-hysteresis-highlat-ecosystem_plot_data/cveg/EC-Earth3-ESM-1_cveg_plot_data.nc
GISS-E2-1-G-CC2 Exp1: 290 years; mean=306.622 Pg C
GISS-E2-1-G-CC2 Exp2: 300 years; mean=312.812 Pg C
GISS-E2-1-G-CC2 Exp4: 97 years; mean=305.719 Pg C
Saved plot data: /mnt/e/lund/NGEA52/hysteresis/tipmip-hysteresis-highlat-ecosystem_plot_data/cveg/GISS-E2-1-G-CC2_cveg_plot_data.nc


/home/jinyu/miniconda3/envs/test-env/lib/python3.11/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


UKESM1-2-LL Exp1: 290 years; mean=51.102 Pg C


/home/jinyu/miniconda3/envs/test-env/lib/python3.11/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


UKESM1-2-LL Exp2: 501 years; mean=52.777 Pg C


/home/jinyu/miniconda3/envs/test-env/lib/python3.11/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


UKESM1-2-LL Exp4: 134 years; mean=39.870 Pg C
Saved plot data: /mnt/e/lund/NGEA52/hysteresis/tipmip-hysteresis-highlat-ecosystem_plot_data/cveg/UKESM1-2-LL_cveg_plot_data.nc
